# Milestone 4 - Sequential Models & CRNNs

## 1) Loading Spectrogram Data

In [14]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import os
import random
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from tqdm import tqdm
import wandb

random.seed(42)
np.random.seed(42)

In [15]:
X = np.load('X_spectrograms_full.npy')
y = np.load('y_labels_full.npy')

X = np.expand_dims(X, axis=1)

enc = LabelEncoder()
y = enc.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

## 2) PyTorch DataLoader

In [16]:
class AudioDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [17]:
train_dataset = AudioDataset(X_train, y_train)
val_dataset = AudioDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [18]:
for batch_X, batch_y in train_loader:
    print(batch_X.shape, batch_y.shape)
    break

torch.Size([32, 1, 128, 1293]) torch.Size([32])


## 3) CRNN Model (CNN + LSTM)

In [ ]:
class AudioCRNN(nn.Module):
    def __init__(self, num_classes=10):
        super(AudioCRNN, self).__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.lstm = nn.LSTM(
            input_size=1024,
            hidden_size=256,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )

        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        batch_size = x.size(0)
        x = x.permute(0, 3, 1, 2)
        x = x.reshape(batch_size, x.size(1), -1)
        lstm_out, _ = self.lstm(x)
        output = self.fc(lstm_out[:, -1, :])
        return output

In [20]:
model = AudioCRNN(num_classes=10)
dummy = torch.randn(32, 1, 128, 1293)
print(model(dummy).shape)

torch.Size([32, 10])


## 4) Training with W&B Logging

In [21]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
model = AudioCRNN(num_classes=10).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0008)

wandb.login(key=os.environ.get('key'))
run = wandb.init(
    project="24f2007692-t12026",
    name='CRNN-spectogram-training-03',
    config={
        "learning_rate": 0.0008,
        "epochs": 50,
        "batch_size": 32,
        "model": "CRNN",
        "architecture": "CNN(4) + LSTM(256)",
    }
)

## 5) Train & Evaluate

In [26]:
EPOCHS = 50

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for features, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        features, labels = features.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(features)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)

    model.eval()
    all_pred, all_labels = [], []

    with torch.no_grad():
        for features, labels in val_loader:
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            _, predicted = torch.max(outputs, 1)

            all_pred.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_pred, average="macro", zero_division=0)
    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {avg_train_loss:.4f} - Val F1: {f1:.4f}")

    wandb.log({"epoch": epoch+1, "train_loss": avg_train_loss, "Macro F1": f1})

print("Training complete!")
wandb.finish()

Epoch 1/50: 100%|██████████| 25/25 [00:03<00:00,  8.16it/s]


Epoch [1/50] - Loss: 2.3240 - Val F1: 0.0182


Epoch 2/50: 100%|██████████| 25/25 [00:02<00:00,  9.96it/s]


Epoch [2/50] - Loss: 2.3008 - Val F1: 0.0880


Epoch 3/50: 100%|██████████| 25/25 [00:02<00:00,  9.91it/s]


Epoch [3/50] - Loss: 2.2702 - Val F1: 0.0747


... (91 outputs left out)

Epoch [49/50] - Loss: 1.2156 - Val F1: 0.4487


Epoch 50/50: 100%|██████████| 25/25 [00:02<00:00,  9.86it/s]


Epoch [50/50] - Loss: 1.1442 - Val F1: 0.4818
Training complete!


Macro F1,▁▂▂▂▂▃▂▃▃▃▃▃▅▅▆▅▅▂▅▆▆▇▆▆▇▆▆▅▆▆▇▅▇▆▆█▇▇▇█
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
train_loss,█████▇▇▇▆▇▆▆▆▅▅▄▄▄▅▄▄▄▄▃▃▃▃▄▃▃▂▄▃▂▂▂▁▂▂▁
Macro F1,0.48183
epoch,50
train_loss,1.14424


## 6) Test Predictions

In [ ]:
test_csv = './messy_mashup/test.csv'
test_folder = './messy_mashup/'
df_test = pd.read_csv(test_csv)

test_files = df_test['filename'].tolist()
id_list = df_test['id'].tolist()

model.eval()
fixed_width = 1293
predictions = []

with torch.no_grad():
    for file in tqdm(test_files, desc="Predicting"):
        file_path = os.path.join(test_folder, file)
        try:
            y, sr = librosa.load(file_path, sr=22050)

            mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, hop_length=512)
            mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)

            if mel_spec_db.shape[1] > fixed_width:
                mel_spec_db = mel_spec_db[:, :fixed_width]
            else:
                pad_width = fixed_width - mel_spec_db.shape[1]
                mel_spec_db = np.pad(mel_spec_db, ((0, 0), (0, pad_width)), mode='constant')

            tensor_audio = torch.tensor(mel_spec_db, dtype=torch.float32)
            tensor_audio = tensor_audio.unsqueeze(0).unsqueeze(0).to(device)

            output = model(tensor_audio)
            predicted_class_idx = torch.argmax(output, dim=1).item()
            predicted_genre = enc.inverse_transform([predicted_class_idx])[0]
            predictions.append(predicted_genre)
        except Exception as e:
            print(e)
            predictions.append('pop')

df_sub = pd.DataFrame({'id': id_list, 'genre': predictions})
df_sub.to_csv('submission.csv', index=False)
print(df_sub.head())